# Land Surface Temperature from Landsat — Tel Aviv

Uses **Landsat Collection 2 Level-2** (Landsat 8 + 9), which includes pre-computed Land Surface Temperature (`ST_B10`) processed by USGS — no manual emissivity estimation needed.

| Item | Detail |
|---|---|
| Source | `LANDSAT/LC08/C02/T1_L2` + `LANDSAT/LC09/C02/T1_L2` |
| LST band | `ST_B10` (scale → Kelvin → Celsius) |
| Resolution | 30 m |
| Output | `tel_aviv_lst_<date>.tif` in Google Drive `GEE_Exports/` |

**Run in:** Google Colab

In [ ]:
!pip install geemap -q

In [ ]:
import ee
import geemap
import datetime

ee.Authenticate()
ee.Initialize(project='your-gee-project-id')  # <-- replace with your GEE project ID

In [ ]:
# ── Study area ───────────────────────────────────────────────────────────────
tel_aviv = ee.Geometry.BBox(34.739, 32.030, 34.840, 32.130)

# ── Date range: last 6 months ────────────────────────────────────────────────
end_date   = datetime.date.today().isoformat()
start_date = (datetime.date.today() - datetime.timedelta(days=180)).isoformat()
print(f'Searching {start_date} → {end_date}')

In [ ]:
# ── Scaling constants for Landsat Collection 2 L2 ST_B10 ────────────────────
# ST_B10 raw → Kelvin: value * 0.00341802 + 149.0
# Kelvin → Celsius: subtract 273.15
LST_SCALE  = 0.00341802
LST_OFFSET = 149.0 - 273.15   # combined: raw → Celsius directly

def apply_lst_scale(image):
    """Convert ST_B10 raw DN to Celsius and add as 'LST' band."""
    lst = (image.select('ST_B10')
               .multiply(LST_SCALE)
               .add(LST_OFFSET)
               .rename('LST'))
    return image.addBands(lst)

def mask_landsat_clouds(image):
    """Mask clouds and cloud shadows using the QA_PIXEL band."""
    qa = image.select('QA_PIXEL')
    # Bit 3 = cloud shadow, Bit 4 = cloud
    cloud_mask = qa.bitwiseAnd(1 << 3).eq(0).And(
                 qa.bitwiseAnd(1 << 4).eq(0))
    return image.updateMask(cloud_mask)

In [ ]:
# ── Load Landsat 8 + 9 Collection 2 Level-2 ─────────────────────────────────
# Merge both sensors for maximum temporal coverage
l8 = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
        .filterBounds(tel_aviv)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUD_COVER', 15))
        .map(mask_landsat_clouds)
        .map(apply_lst_scale))

l9 = (ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
        .filterBounds(tel_aviv)
        .filterDate(start_date, end_date)
        .filter(ee.Filter.lt('CLOUD_COVER', 15))
        .map(mask_landsat_clouds)
        .map(apply_lst_scale))

collection = l8.merge(l9).sort('system:time_start', False)

print(f'Landsat 8 images : {l8.size().getInfo()}')
print(f'Landsat 9 images : {l9.size().getInfo()}')
print(f'Total combined   : {collection.size().getInfo()}')

In [ ]:
# ── Pick most recent single image ────────────────────────────────────────────
most_recent = collection.first()

date_str    = ee.Date(most_recent.get('system:time_start')).format('YYYY-MM-dd').getInfo()
cloud_pct   = most_recent.get('CLOUD_COVER').getInfo()
satellite   = most_recent.get('SPACECRAFT_ID').getInfo()
print(f'Most recent image : {date_str}')
print(f'Satellite         : {satellite}')
print(f'Cloud cover       : {cloud_pct:.1f}%')

lst = most_recent.select('LST').clip(tel_aviv)

In [ ]:
# ── Print LST statistics over Tel Aviv ──────────────────────────────────────
stats = lst.reduceRegion(
    reducer=ee.Reducer.minMax().combine(ee.Reducer.mean(), sharedInputs=True),
    geometry=tel_aviv,
    scale=30,
    maxPixels=1e9
).getInfo()

print(f"LST min  : {stats['LST_min']:.1f} °C")
print(f"LST max  : {stats['LST_max']:.1f} °C")
print(f"LST mean : {stats['LST_mean']:.1f} °C")

In [ ]:
# ── Interactive map preview ──────────────────────────────────────────────────
Map = geemap.Map(center=[32.08, 34.79], zoom=12)

lst_vis = {
    'min': 20,
    'max': 50,
    'palette': ['#313695', '#4575b4', '#74add1', '#abd9e9',
                '#fee090', '#fdae61', '#f46d43', '#d73027', '#a50026']
}

Map.addLayer(lst, lst_vis, f'LST °C  {date_str}  ({satellite})')
Map.add_colorbar(lst_vis, label='LST (°C)', layer_name=f'LST °C  {date_str}  ({satellite})')
Map

In [ ]:
# ── Export to Google Drive ───────────────────────────────────────────────────
export_task = ee.batch.Export.image.toDrive(
    image=lst,
    description=f'tel_aviv_lst_{date_str}',
    folder='GEE_Exports',
    fileNamePrefix=f'tel_aviv_lst_{date_str}',
    region=tel_aviv,
    scale=30,
    crs='EPSG:4326',
    maxPixels=1e9,
    fileFormat='GeoTIFF'
)

export_task.start()
print('Export started.')
print(f'Output → Google Drive/GEE_Exports/tel_aviv_lst_{date_str}.tif')

In [ ]:
# ── (Optional) Monitor export status ────────────────────────────────────────
import time

while export_task.active():
    status = export_task.status()
    print(f"State: {status['state']}  |  {datetime.datetime.now().strftime('%H:%M:%S')}")
    time.sleep(30)

print(f"Final state: {export_task.status()['state']}")

## After the Export Finishes

1. Download `tel_aviv_lst_<date>.tif` from Google Drive.
2. Place it in `Layers/` in this project.
3. Load it in subsequent notebooks:

```python
import rasterio
import numpy as np

with rasterio.open('Layers/tel_aviv_lst_<date>.tif') as src:
    lst_array = src.read(1)   # values in °C
    transform = src.transform
    crs       = src.crs
```

## Notes

- LST is from a **single overpass** (~10:30 local time for Landsat). It reflects daytime surface temperature, not air temperature.
- For a more robust estimate, generate a **median composite** over summer months by replacing `collection.first()` with `collection.median()` before selecting the LST band.
- Landsat revisit time over Tel Aviv is ~16 days, so 6 months gives ~10–12 candidate scenes.
- NDVI (10 m) and LST (30 m) have different resolutions — resample to a common grid (e.g. 30 m or 100 m) when combining them for the heat priority score.